In [5]:
pip install pygame

In [2]:
import numpy as np
import pygame

pygame 2.6.1 (SDL 2.28.4, Python 3.11.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [7]:
 class Connect4:

    def __init__(self):
        self.filas = 6
        self.columnas = 7
        self.tablero = [[0 for _ in range(self.columnas)] for _ in range(self.filas)]
        print("Inicializando partida de 'Conecta 4'...\n")
        print("Ficha del jugador: 1 \nFicha de la máquina: 2 \n")
        self.imprimir_tablero()

    # -----------------------------
    # imprimir el tablero
    # -----------------------------
    def imprimir_tablero(self):
        for fila in self.tablero:
            print("| " + " ".join(str(x) for x in fila) + " |")
        print("  " + " ".join(str(i) for i in range(self.columnas)))
        print()

    # -----------------------------
    # chequear si está el tablero lleno
    # -----------------------------
    def tablero_lleno(self, tablero):
        return all(tablero[0][col] != 0 for col in range(self.columnas))

    # -----------------------------
    # copiar el tablero (para no cambiarlo y analizar posibles jugadas)
    # -----------------------------
    def copiar_tablero(self, tablero):
        return [fila[:] for fila in tablero]

    # -----------------------------
    # colocar la ficha
    # -----------------------------
    def colocar_ficha(self, tablero, col, ficha):
        for fila in range(self.filas - 1, -1, -1):
            if tablero[fila][col] == 0:
                tablero[fila][col] = ficha
                return fila
        return None  # columna llena

    # -----------------------------
    # chequear si hay ganador
    # -----------------------------
    def hay_ganador(self, tablero):
        # horizontales
        for f in range(self.filas):
            for c in range(self.columnas - 3):
                linea = [tablero[f][c+i] for i in range(4)]
                if linea == [1]*4:
                    return 1
                if linea == [2]*4:
                    return 2

        # verticales
        for f in range(self.filas - 3):
            for c in range(self.columnas):
                linea = [tablero[f+i][c] for i in range(4)]
                if linea == [1]*4:
                    return 1
                if linea == [2]*4:
                    return 2

        # diagonal positiva
        for f in range(self.filas - 3):
            for c in range(self.columnas - 3):
                linea = [tablero[f+i][c+i] for i in range(4)]
                if linea == [1]*4:
                    return 1
                if linea == [2]*4:
                    return 2

        # diagonal negativa
        for f in range(3, self.filas):
            for c in range(self.columnas - 3):
                linea = [tablero[f-i][c+i] for i in range(4)]
                if linea == [1]*4:
                    return 1
                if linea == [2]*4:
                    return 2

        return None

    # -----------------------------
    # función evaluación 1
    # -----------------------------
    def evaluar_tablero1(self, tablero):
        puntuacion = 0

        def evaluar_linea1(linea):
            nonlocal puntuacion
            if linea.count(2) == 4:
                puntuacion += 1000
            elif linea.count(1) == 4:
                puntuacion -= 1000
            elif linea.count(2) == 3 and linea.count(0) == 1:
                puntuacion += 10
            elif linea.count(1) == 3 and linea.count(0) == 1:
                puntuacion -= 10
            elif linea.count(2) == 2 and linea.count(0) == 2:
                puntuacion += 1
            elif linea.count(1) == 2 and linea.count(0) == 2:
                puntuacion -= 1

        # horizontales
        for f in range(self.filas):
            for c in range(self.columnas - 3):
                evaluar_linea1([tablero[f][c+i] for i in range(4)])

        # verticales
        for f in range(self.filas - 3):
            for c in range(self.columnas):
                evaluar_linea1([tablero[f+i][c] for i in range(4)])

        # diagonal positiva
        for f in range(self.filas - 3):
            for c in range(self.columnas - 3):
                evaluar_linea1([tablero[f+i][c+i] for i in range(4)])

        # diagonal negativa
        for f in range(3, self.filas):
            for c in range(self.columnas - 3):
                evaluar_linea1([tablero[f-i][c+i] for i in range(4)])

        return puntuacion

    # -----------------------------
    # función evaluación 2
    # -----------------------------    
    
    def evaluar_tablero2(self, tablero):
        
        def es_jugable(tablero, fila, col):
            # Una celda es jugable si está vacía y:
            # - está en la última fila, o
            # - la celda debajo está ocupada
            return tablero[fila][col] == 0 and (fila == self.filas-1 or tablero[fila+1][col] != 0)
            
        def evaluar_linea2(linea, tablero, coords):
            puntuacion = 0
            maquina = 1
            jugador = -1
        
            count_maquina = linea.count(maquina)
            count_jugador = linea.count(jugador)
            count_vacios = linea.count(0)
        
            # Obtener posiciones de vacíos
            vacios_coords = [coords[i] for i in range(4) if linea[i] == 0]
        
            # Comprobar si los vacíos son jugables
            jugables = sum(1 for (f, c) in vacios_coords if es_jugable(tablero, f, c))
        
            # Evaluación mejorada
            if count_maquina == 4:
                puntuacion += 10000
                
            elif count_jugador == 4:
                puntuacion -= 10000
        
            elif count_maquina == 3 and count_vacios == 1 and jugables == 1:
                puntuacion += 50
                
            elif count_jugador == 3 and count_vacios == 1 and jugables == 1:
                puntuacion -= 80  # más peso defensivo
        
            elif count_maquina == 2 and count_vacios == 2:
                puntuacion += 5
                
            elif count_jugador == 2 and count_vacios == 2:
                puntuacion -= 5
        
            return puntuacion

        
        puntuacion = 0
    
        # Ponderación del control del centro
        col_central = self.columnas // 2
        for f in range(self.filas):
            if tablero[f][col_central] == 1:
                puntuacion += 6
            elif tablero[f][col_central] == -1:
                puntuacion -= 6

        # Evaluar todas las líneas
        # Horizontal
        for f in range(self.filas):
            for c in range(self.columnas - 3):
                coords = [(f, c+i) for i in range(4)]
                linea = [tablero[f][c+i] for i in range(4)]
                puntuacion += evaluar_linea2(linea, tablero, coords)
    
        # Vertical
        for f in range(self.filas - 3):
            for c in range(self.columnas):
                coords = [(f+i, c) for i in range(4)]
                linea = [tablero[f+i][c] for i in range(4)]
                puntuacion += evaluar_linea2(linea, tablero, coords)
    
        # Diagonal positiva
        for f in range(self.filas - 3):
            for c in range(self.columnas - 3):
                coords = [(f+i, c+i) for i in range(4)]
                linea = [tablero[f+i][c+i] for i in range(4)]
                puntuacion += evaluar_linea2(linea, tablero, coords)
    
        # Diagonal negativa
        for f in range(3, self.filas):
            for c in range(self.columnas - 3):
                coords = [(f-i, c+i) for i in range(4)]
                linea = [tablero[f-i][c+i] for i in range(4)]
                puntuacion += evaluar_linea2(linea, tablero, coords)
    
        return puntuacion

    # -----------------------------
    # minimax con poda alpha-beta
    # -----------------------------
    def minimax(self, tablero, profundidad, alpha, beta, maximizar):
        if profundidad == 0 or self.hay_ganador(tablero) or self.tablero_lleno(tablero):
            return self.evaluar_tablero2(tablero), None

        if maximizar: # si estamos maximizando (jugada de la máquina)
            max_eval = float('-inf')
            best_col = None

            for col in range(self.columnas):
                if tablero[0][col] != 0:
                    continue
                # para cada columna mira si está llena y en caso de no estarlo explora esa jugada:
                copia = self.copiar_tablero(tablero)
                self.colocar_ficha(copia, col, 2) # coloca a ficha de la máquina en la columna no llena

                eval_score, _ = self.minimax(copia, profundidad-1, alpha, beta, False) # la idea es ver que jugada del jugador minimiza la puntuación
                # para observar que jugada anterior que haga la máquina es más beneficiosa para ella

                if eval_score > max_eval: # compara la evaluación obtenida con la máxima anterior y actualiza en caso de ser mayor
                    max_eval = eval_score
                    best_col = col

                alpha = max(alpha, eval_score) # actualiza alpha con el mayor valor del nivel (beneficio para la máquina)
                if beta <= alpha: 
                    break

            return max_eval, best_col # devuelve la mejor jugada para la máquina en función de la posible siguiente mejor jugada del jugador

        else: # si estamos minimizando (previendo la jugada del jugador)
            min_eval = float('inf')
            best_col = None

            for col in range(self.columnas):
                if tablero[0][col] != 0:
                    continue
                # para cada columna mira si está llena y en caso de no estarlo explora esa jugada:
                copia = self.copiar_tablero(tablero)
                self.colocar_ficha(copia, col, 1) # coloca a ficha del jugador en la columna no llena

                eval_score, _ = self.minimax(copia, profundidad-1, alpha, beta, True) # en el caso de proponer mayor profundidad volvería a explorar las 
                # posibles siguientes jugadas de la máquina

                if eval_score < min_eval: # compara la evaluación obtenida con la mínima anterior y actualiza en caso de ser menor
                    min_eval = eval_score
                    best_col = col

                beta = min(beta, eval_score) # actualiza beta con el menor valor del nivel (beneficio para el jugador)
                if beta <= alpha:
                    break

            return min_eval, best_col # devuelve la mejor jugada para el jugador en función de la posible siguiente mejor jugada de la máquina
            # (no en el caso básico de profundidad=2)

    # -----------------------------
    # obtener el mejor movimiento para la máquina (inicializa la función minimax)
    # -----------------------------
    def mejor_movimiento(self, tablero):
        _, col = self.minimax(tablero, profundidad=2,
                              alpha=float('-inf'),
                              beta=float('inf'),
                              maximizar=True)
        return col # devuelve la columna en la que debe colocar la ficha la máquina para máximixar su puntuación (probabilidad de ganar)

    # -----------------------------
    # recursión del juego
    # -----------------------------
    def connect4(self):
        tablero_actual = self.tablero

        # chequear ganador
        ganador = self.hay_ganador(tablero_actual)
        if ganador:
            self.imprimir_tablero()
            print("El jugador ha ganado!" if ganador == 1 else "La máquina ha ganado!")
            return

        # checquear si el tablero está lleno
        if self.tablero_lleno(tablero_actual):
            self.imprimir_tablero()
            print("Empate: tablero lleno")
            return

        # movimiento del jugador
        mov_jug = int(input("Jugador, ¿en qué columna coloca la ficha? "))
        self.colocar_ficha(tablero_actual, mov_jug, 1)

        # actualizar tablero
        self.tablero = tablero_actual

        # chequear si el jugador ha ganado o el tablero está lleno
        ganador = self.hay_ganador(tablero_actual)
        if ganador:
            self.imprimir_tablero()
            print("El jugador ha ganado!")
            return

        if self.tablero_lleno(tablero_actual):
            self.imprimir_tablero()
            print("Empate: tablero lleno")
            return

        # imprimir tablero con el nuevo movimiento del jugador
        self.imprimir_tablero()

        # movimiento de la máquina
        mov_maq = self.mejor_movimiento(tablero_actual)
        self.colocar_ficha(tablero_actual, mov_maq, 2)

        self.tablero = tablero_actual

        # imprimir el tablero
        self.imprimir_tablero()

        # siguiente jugada
        return self.connect4()
        
game = Connect4()
game.connect4()

Inicializando partida de 'Conecta 4'...

Ficha del jugador: 1 
Ficha de la máquina: 2 

| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
  0 1 2 3 4 5 6



Jugador, ¿en qué columna coloca la ficha?  3


| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 1 0 0 0 |
  0 1 2 3 4 5 6

| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 2 0 0 0 |
| 0 0 0 1 0 0 0 |
  0 1 2 3 4 5 6



Jugador, ¿en qué columna coloca la ficha?  4


| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 2 0 0 0 |
| 0 0 0 1 1 0 0 |
  0 1 2 3 4 5 6

| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 0 0 0 0 |
| 0 0 0 2 0 0 0 |
| 0 0 0 2 0 0 0 |
| 0 0 0 1 1 0 0 |
  0 1 2 3 4 5 6



...

| 0 0 0 0 0 0 0 |
| 0 0 0 2 0 0 0 |
| 0 0 0 1 0 0 0 |
| 0 0 0 2 0 0 0 |
| 0 0 0 2 1 0 0 |
| 0 0 0 1 1 0 0 |
  0 1 2 3 4 5 6

| 0 0 0 2 0 0 0 |
| 0 0 0 2 0 0 0 |
| 0 0 0 1 0 0 0 |
| 0 0 0 2 0 0 0 |
| 0 0 0 2 1 0 0 |
| 0 0 0 1 1 0 0 |
  0 1 2 3 4 5 6



Jugador, ¿en qué columna coloca la ficha?  1


| 0 0 0 2 0 0 0 |
| 0 0 0 2 0 0 0 |
| 0 0 0 1 0 0 0 |
| 0 0 0 2 0 0 0 |
| 0 0 0 2 1 0 0 |
| 0 1 0 1 1 0 0 |
  0 1 2 3 4 5 6

| 0 0 0 2 0 0 0 |
| 0 0 0 2 0 0 0 |
| 0 0 0 1 0 0 0 |
| 0 0 0 2 2 0 0 |
| 0 0 0 2 1 0 0 |
| 0 1 0 1 1 0 0 |
  0 1 2 3 4 5 6



Jugador, ¿en qué columna coloca la ficha?  2


| 0 0 0 2 0 0 0 |
| 0 0 0 2 0 0 0 |
| 0 0 0 1 0 0 0 |
| 0 0 0 2 2 0 0 |
| 0 0 0 2 1 0 0 |
| 0 1 1 1 1 0 0 |
  0 1 2 3 4 5 6

El jugador ha ganado!
